In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# The \"ollmcp\" Native Bridge for Colab\n",
    "\n",
    "Since we cannot run the interactive `ollmcp` TUI in Colab, this script implements the same logic in pure Python.\n",
    "\n",
    "**Capabilities:**\n",
    "1. Starts Ollama (GPU Accelerated)\n",
    "2. Runs `code-scalpel` as an MCP Server via `uvx`\n",
    "3. **Dynamically Grounds** the LLM by injecting valid tool names into the system prompt to prevent hallucinations."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Install Dependencies\n",
    "We need `pciutils` for the T4 GPU, `uv` for the tool runner, and the `mcp`/`ollama` SDKs."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# System Deps\n",
    "!apt-get update && apt-get install -y pciutils lshw\n",
    "\n",
    "# Python Deps\n",
    "!pip install uv mcp ollama nest_asyncio"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Start Ollama & Download Model\n",
    "We use `qwen2.5-coder:7b`. It is the best balance of speed and tool-use capability for a T4 GPU."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import subprocess\n",
    "import threading\n",
    "import time\n",
    "\n",
    "# 1. Install & Start Ollama\n",
    "!curl -fsSL https://ollama.com/install.sh | sh\n",
    "\n",
    "def start_server():\n",
    "    with open(\"ollama.log\", \"w\") as f:\n",
    "        subprocess.run([\"ollama\", \"serve\"], stdout=f, stderr=f)\n",
    "\n",
    "threading.Thread(target=start_server, daemon=True).start()\n",
    "time.sleep(10)\n",
    "\n",
    "# 2. Pull Model\n",
    "!ollama pull qwen2.5-coder:7b"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: Create the \"Proving Ground\" File\n",
    "We create a Python file with messy imports and no type hints to test the agent."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "messy_code = \"\"\"\n",
    "import sys, os\n",
    "import json\n",
    "from datetime import datetime\n",
    "\n",
    "def calculate(a, b):\n",
    "    return a + b\n",
    "\n",
    "def sensitive_query(user_input):\n",
    "    # This is vulnerable to SQL Injection\n",
    "    q = \"SELECT * FROM users WHERE name = \" + user_input\n",
    "    return q\n",
    "\"\"\"\n",
    "\n",
    "with open(\"proving_ground.py\", \"w\") as f:\n",
    "    f.write(messy_code)\n",
    "\n",
    "print(\"✅ Created 'proving_ground.py'\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: The Intelligent Agent Loop\n",
    "This is the core script. It connects to `uvx`, reads the tools, and forces the LLM to use them correctly."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import asyncio\n",
    "import os\n",
    "import ollama\n",
    "import nest_asyncio\n",
    "from mcp import ClientSession, StdioServerParameters\n",
    "from mcp.client.stdio import stdio_client\n",
    "\n",
    "nest_asyncio.apply()\n",
    "\n",
    "async def run_agent():\n",
    "    # --- CONFIGURATION ---\n",
    "    model_id = \"qwen2.5-coder:7b\"\n",
    "    \n",
    "    # Define the MCP Server (Code Scalpel via uvx)\n",
    "    server_params = StdioServerParameters(\n",
    "        command=\"uvx\",\n",
    "        args=[\"codescalpel\", \"mcp\"],\n",
    "        env={**os.environ}\n",
    "    )\n",
    "    \n",
    "    print(\"🔌 Connecting to Code Scalpel MCP Server...\")\n",
    "    \n",
    "    # Use a log file to capture stderr without blocking Colab\n",
    "    log_file = open(\"mcp_debug.log\", \"wb\")\n",
    "\n",
    "    async with stdio_client(server_params, errlog=log_file) as (read, write):\n",
    "        async with ClientSession(read, write) as session:\n",
    "            await session.initialize()\n",
    "            \n",
    "            # 1. DISCOVER TOOLS\n",
    "            mcp_tools = await session.list_tools()\n",
    "            \n",
    "            # Convert to Ollama format\n",
    "            ollama_tools = []\n",
    "            tool_names = []\n",
    "            for t in mcp_tools.tools:\n",
    "                tool_names.append(t.name)\n",
    "                ollama_tools.append({\n",
    "                    'type': 'function',\n",
    "                    'function': {\n",
    "                        'name': t.name,\n",
    "                        'description': t.description,\n",
    "                        'parameters': t.inputSchema\n",
    "                    }\n",
    "                })\n",
    "            \n",
    "            print(f\"✅ Connected! Discovered {len(ollama_tools)} tools.\")\n",
    "            print(f\"📜 AVAILABLE TOOLS: {', '.join(tool_names)}\")\n",
    "            \n",
    "            # 2. DEFINE THE CHAT LOGIC\n",
    "            async def chat(user_prompt):\n",
    "                print(f\"\\n🔵 USER: {user_prompt}\")\n",
    "                \n",
    "                # --- GROUNDING SYSTEM PROMPT ---\n",
    "                # This is the secret sauce. We explicitly list the tools in the system prompt.\n",
    "                # This prevents the model from guessing names like 'reorder_imports'.\n",
    "                system_prompt = (\n",
    "                    f\"You are an expert coding agent. \"\n",
    "                    f\"You have access to the following tools: {tool_names}. \"\n",
    "                    f\"You MUST use these tools to answer the user's request. \"\n",
    "                    f\"Do not Hallucinate tool names. Only use the names listed above.\"\n",
    "                )\n",
    "\n",
    "                messages = [\n",
    "                    {'role': 'system', 'content': system_prompt},\n",
    "                    {'role': 'user', 'content': user_prompt}\n",
    "                ]\n",
    "\n",
    "                # Call Ollama\n",
    "                response = ollama.chat(\n",
    "                    model=model_id,\n",
    "                    messages=messages,\n",
    "                    tools=ollama_tools\n",
    "                )\n",
    "\n",
    "                # Check for Tool Calls\n",
    "                if response.message.tool_calls:\n",
    "                    messages.append(response.message)\n",
    "                    \n",
    "                    for tool_call in response.message.tool_calls:\n",
    "                        fn_name = tool_call.function.name\n",
    "                        fn_args = tool_call.function.arguments\n",
    "                        \n",
    "                        print(f\"🔶 AGENT DECISION: Call '{fn_name}'\")\n",
    "                        print(f\"   ARGS: {fn_args}\")\n",
    "                        \n",
    "                        # EXECUTE via MCP\n",
    "                        try:\n",
    "                            result = await session.call_tool(fn_name, arguments=fn_args)\n",
    "                            \n",
    "                            # Get text output\n",
    "                            output_text = \"\".join([c.text for c in result.content if c.type == 'text'])\n",
    "                            print(f\"✅ TOOL OUTPUT: {output_text[:100]}... (truncated)\")\n",
    "                            \n",
    "                            messages.append({'role': 'tool', 'content': output_text})\n",
    "                            \n",
    "                        except Exception as e:\n",
    "                            print(f\"🔴 TOOL ERROR: {e}\")\n",
    "                            messages.append({'role': 'tool', 'content': f\"Error: {str(e)}\"})\n",
    "\n",
    "                    # Final Response from LLM\n",
    "                    final_response = ollama.chat(\n",
    "                        model=model_id,\n",
    "                        messages=messages\n",
    "                    )\n",
    "                    print(f\"⚪ FINAL ANSWER: {final_response.message.content}\")\n",
    "                else:\n",
    "                    print(f\"⚪ FINAL ANSWER: {response.message.content}\")\n",
    "\n",
    "            # 3. RUN SCENARIOS\n",
    "            # Analysis\n",
    "            await chat(\"Use code-scalpel to analyze 'proving_ground.py' for structure and issues.\")\n",
    "            \n",
    "            # Security Scan (looking for that SQL injection)\n",
    "            await chat(\"Scan 'proving_ground.py' for security vulnerabilities.\")\n",
    "            \n",
    "            # Formatting/Refactoring\n",
    "            # Note: We ask it to check the list first to avoid 'reorder_imports' hallucination\n",
    "            await chat(\"Please organize the imports in 'proving_ground.py' using the appropriate tool from your list.\")\n",
    "\n",
    "if __name__ == '__main__':\n",
    "    asyncio.run(run_agent())"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}